# IAAIS Chapter 4 — Planner

The Planner reads the current symbolic state from the Knowledge Base, uses the Search Engine to find a goal-directed action sequence, and writes the proposed plan back to the Knowledge Base.

For the current IAAIS exercise-recording domain, planning is deterministic: the system is organizing a finite recorded session rather than controlling a physical agent with stochastic outcomes. The deterministic Planner therefore uses forward state-space search with a delete-relaxed heuristic. A generic value-iteration component is included for a future domain with stochastic actions.

In [ ]:
from iaais.knowledge_base import KnowledgeBase, P
from iaais.planner import GoalSpec, Planner, PlanningAction, ValueIterationPlanner, StochasticAction, StochasticOutcome, fact_key

## Deterministic planning from the Knowledge Base

The actions below are information-processing actions. They do not prescribe exercise or control a client. They demonstrate how the Planner can transform a current session fact into a reviewable workout-log goal.

In [ ]:
kb = KnowledgeBase()
kb.assert_fact("session_open", ("S1",), fact_id="session-open", source="session")

session_open = fact_key("session_open", ("S1",))
context_ready = fact_key("context_ready", ("S1",))
segment_ready = fact_key("segment_ready", ("S1", "SEG1"))
workout_logged = fact_key("workout_logged", ("S1",))

actions = (
    PlanningAction("open-session-context", frozenset({session_open}), frozenset({context_ready}), cost=1),
    PlanningAction("prepare-segment", frozenset({context_ready}), frozenset({segment_ready}), cost=1),
    PlanningAction("record-workout", frozenset({segment_ready}), frozenset({workout_logged}), cost=1),
)
goal = GoalSpec(positive=frozenset({workout_logged}))
planner = Planner(kb)
plan = planner.plan(actions, goal, plan_id="plan-demo")

print("status:", plan.status.value)
print("actions:", [action.name for action in plan.actions])
print("cost:", plan.total_cost)
print("optimality guaranteed:", plan.optimality_guaranteed)

In [ ]:
stored_plan = kb.query(P("plan_result", "plan-demo", "success", tuple(action.name for action in plan.actions)))
print("plan stored in KB:", stored_plan.status.value)

execution_records = planner.apply_execution_update(plan.actions[-1], execution_id="execution-demo")
print("execution records:", [record.fact_id for record in execution_records])

## Review and replanning after corrected evidence

A proposed sensor interpretation may support a provisional plan, but it cannot silently become confirmed truth. If a trainer corrects the interpretation, the Knowledge Base retracts the old evidence and the Planner replans from the corrected state.

In [ ]:
from iaais.knowledge_base import FactStatus, Polarity

review_kb = KnowledgeBase()
review_kb.assert_fact(
    "exercise_candidate",
    ("SEG1", "back_squat"),
    fact_id="candidate-squat",
    status=FactStatus.PROPOSED,
    source="exercise-model",
)
candidate_squat = fact_key("exercise_candidate", ("SEG1", "back_squat"))
candidate_lunge = fact_key("exercise_candidate", ("SEG1", "lunge"))
review_goal_fact = fact_key("reviewed_workout_logged", ("S1",))
review_actions = (
    PlanningAction("log-squat", frozenset({candidate_squat}), frozenset({review_goal_fact})),
    PlanningAction("log-lunge", frozenset({candidate_lunge}), frozenset({review_goal_fact})),
)
review_planner = Planner(review_kb)
first_review_plan = review_planner.plan(
    review_actions,
    GoalSpec(positive=frozenset({review_goal_fact})),
    plan_id="plan-before-correction",
)
print("initial action:", first_review_plan.actions[0].name)
print("requires review:", first_review_plan.requires_review)

review_kb.retract_fact(
    "candidate-squat",
    reason="Trainer corrected the exercise interpretation",
)
review_kb.assert_fact(
    "exercise_candidate",
    ("SEG1", "lunge"),
    fact_id="candidate-lunge",
    status=FactStatus.CONFIRMED,
    source="trainer-review",
)
second_review_plan = review_planner.replan(
    review_actions,
    GoalSpec(positive=frozenset({review_goal_fact})),
    previous_plan_id=first_review_plan.plan_id,
    reason="Replan after trainer correction",
)
print("corrected action:", second_review_plan.actions[0].name)
print("replanned from:", second_review_plan.replanned_from)

In [ ]:
conflict_kb = KnowledgeBase()
conflict_kb.assert_fact("exercise_candidate", ("SEG2", "deadlift"), fact_id="positive")
conflict_kb.assert_fact(
    "exercise_candidate",
    ("SEG2", "deadlift"),
    fact_id="negative",
    polarity=Polarity.NEGATIVE,
)
conflict_fact = fact_key("exercise_candidate", ("SEG2", "deadlift"))
blocked_plan = Planner(conflict_kb).plan(
    (PlanningAction("log-deadlift", frozenset({conflict_fact}), frozenset({review_goal_fact})),),
    GoalSpec(positive=frozenset({review_goal_fact})),
    plan_id="plan-conflict",
)
print("conflicting-evidence status:", blocked_plan.status.value)
print("review required:", blocked_plan.requires_review)

## Optional stochastic planning

Value iteration returns a policy rather than one guaranteed action sequence. This is appropriate only when actions have probabilistic outcomes and the domain has a meaningful reward function.

In [ ]:
states = ("start", "goal", "fail")

def uncertain_actions(state):
    if state == "start":
        return (StochasticAction("try-action", (StochasticOutcome("goal", 0.8, 1), StochasticOutcome("fail", 0.2, -1))),)
    return ()

mdp = ValueIterationPlanner(states, uncertain_actions, terminal_states=("goal", "fail"))
mdp_result = mdp.solve()
print("converged:", mdp_result.converged)
print("policy at start:", mdp_result.policy["start"].name)

The deterministic Planner is the active IAAIS path. It uses the Knowledge Base as the current-world source, the Search Engine as the path generator, and the Knowledge Base again as the plan-result store. Later Decision Agent work can report execution updates through the Planner's symbolic update boundary.